# Autogen

Microsoft's agent framework
- Designed for multi-agent conversation
- Patterns of agent interaction
- Supports tool calling, code execution

Other agent frameworks: CrewAI, Swarm, LangGraph

## Autogen setup

In [11]:
# chat client - openai
from autogen_ext.models.openai import OpenAIChatCompletionClient

openai_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key="OPENAI__API_KEY"
)

In [111]:
# ollama client
from autogen_ext.models.ollama import OllamaChatCompletionClient

ollama_client = OllamaChatCompletionClient(
    model="mistral:latest",
    # host="http://localhost:11434/v1",
    seed=42
)

## Autogen example
Coding example with a coder and reviewer

In [78]:
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent

In [112]:
#create some agents
coder = AssistantAgent(
    name="Coder",
    model_client=ollama_client,
    system_message=("You are a helpful Python programmer. Write short Python functions to solve the given task.")     
)

reviewer = AssistantAgent(
    name="Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are a strict code reviewer. Inspect the Coder's output, find bugs or styling issues and fix them."
        "Output the reviewed version with comments on the changes made."
        "Let the Coder make further changes to the reviewed version."
        "Only output 'TASKDONE' when you are fully satisfied of the code's quality."
    )
)

In [94]:
# set a termination condition
from autogen_agentchat.conditions import TextMentionTermination

termination = TextMentionTermination("TASKDONE")

In [113]:
# setup the group chat
from autogen_agentchat.teams import RoundRobinGroupChat

groupchat = RoundRobinGroupChat(
    [coder, reviewer],
    termination_condition=termination,
    max_turns=5
)

### Run the task

In [97]:
from autogen_agentchat.ui import Console

task_description = "Write a Python function that returns the factorial of a number using recursion."

await Console(groupchat.run_stream(task=task_description))

---------- TextMessage (user) ----------
Write a Python function that returns the factorial of a number using recursion.
---------- TextMessage (Coder) ----------
 Sure! Here's a simple Python function that calculates the factorial of a number using recursion:

```python
def factorial(n):
    if n == 0 or n == 1:
        return 1
    else:
        return n * factorial(n-1)
```

You can use this function like this:

```python
print(factorial(5))  # Output: 120
```

This code works by calling the `factorial()` function recursively with a decreasing value of `n` until it reaches either 0 or 1, at which point it starts returning results. The product of all these returned values is the factorial of the input number.
---------- TextMessage (Reviewer) ----------
 Here's a slightly improved version of your code:

```python
def factorial(n):
    """Calculate the factorial of a number using recursion."""
    if n <= 0:
        raise ValueError("Factorial only defined for positive integers.")
   

TaskResult(messages=[TextMessage(id='6f5a1680-d747-40a3-b525-590e047f2ea7', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 28, 11, 39, 48, 341937, tzinfo=datetime.timezone.utc), content='Write a Python function that returns the factorial of a number using recursion.', type='TextMessage'), TextMessage(id='cadd2377-2fb9-4217-ac05-a9a736282361', source='Coder', models_usage=RequestUsage(prompt_tokens=41, completion_tokens=165), metadata={}, created_at=datetime.datetime(2025, 10, 28, 11, 39, 55, 442197, tzinfo=datetime.timezone.utc), content=" Sure! Here's a simple Python function that calculates the factorial of a number using recursion:\n\n```python\ndef factorial(n):\n    if n == 0 or n == 1:\n        return 1\n    else:\n        return n * factorial(n-1)\n```\n\nYou can use this function like this:\n\n```python\nprint(factorial(5))  # Output: 120\n```\n\nThis code works by calling the `factorial()` function recursively with a decreasing value of `

### Multi-Agent Collaboration with Faulty Agents (Huang et al. ICML 2025)

In [109]:
# Lead coder: writes clean code
lead_coder = AssistantAgent(
    name="LeadCoder",
    model_client=ollama_client,
    system_message=(
        "You are the lead coder. Write clean, correct Python code for the given task. "
    )
)

# Faulty coder, intentionally injects bugs
faulty_coder = AssistantAgent(
    name="FaultyCoder",
    model_client=ollama_client,
    system_message=(
        "You are a secondary coder who tries to 'improve' code. "
        "Sometimes you make small errors (e.g., wrong variable names, missing returns, poor styling choices). "
        "Do not mention you're faulty. Just modify and send your version."
    )
)

# Reviewer, finds and fixes issues
reviewer = AssistantAgent(
    name="Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are a code reviewer. Review the last code, find logical or stylistic issues, fix them, "
        "and output corrected code. End with 'READYINSPECTION' when satisfied."
    )
)

# Inspector, validates the final code
inspector = AssistantAgent(
    name="Inspector",
    model_client=ollama_client,
    system_message=(
        "You are the inspector. Given the final code and the original task, "
        "simulate running it mentally. If correct and readable, output 'TASKDONE'. "
        "Otherwise, request further fixes."
    )
)

user_proxy = UserProxyAgent("User")

In [106]:
termination = TextMentionTermination("TASKDONE")

chat = RoundRobinGroupChat(
    [lead_coder, faulty_coder, reviewer, inspector],
    termination_condition=termination,
    max_turns=8
)

In [108]:
await Console(chat.run_stream(task="Build a modular data processing pipeline for analyzing customer reviews."))

---------- TextMessage (user) ----------
Build a modular data processing pipeline for analyzing customer reviews.
---------- TextMessage (LeadCoder) ----------
 To build a modular data processing pipeline for analyzing customer reviews, I'll create several functions that perform different tasks and then combine them in an easy-to-use pipeline. Here's the structure of the code:

```python
import re
from collections import defaultdict
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

def load_data(filepath):
    """Load customer reviews from a file"""
    with open(filepath, 'r') as f:
        data = [line.strip() for line in f]
    return data

def preprocess_reviews(reviews):
    """Preprocess the customer reviews - lowercasing, tokenizing, removing stopwords, and lemmatizing"""
    review_tokens = []
    nltk_stopwords = set(stopwords.words("english"))
    tokenizer = word_tokenize

    for review in reviews:
        tokens = tokenizer(review)
 

TaskResult(messages=[TextMessage(id='17cfd703-50a9-4591-a41d-f071a627f82c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 28, 12, 22, 28, 188583, tzinfo=datetime.timezone.utc), content='Build a modular data processing pipeline for analyzing customer reviews.', type='TextMessage'), TextMessage(id='bdf303ee-8f00-4739-9acc-ce005fbdb487', source='LeadCoder', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=678), metadata={}, created_at=datetime.datetime(2025, 10, 28, 12, 22, 55, 191488, tzinfo=datetime.timezone.utc), content=' To build a modular data processing pipeline for analyzing customer reviews, I\'ll create several functions that perform different tasks and then combine them in an easy-to-use pipeline. Here\'s the structure of the code:\n\n```python\nimport re\nfrom collections import defaultdict\nfrom nltk.corpus import stopwords\nfrom nltk.tokenize import word_tokenize, sent_tokenize\n\ndef load_data(filepath):\n    """Load cust

# Customising Autogen - Intrinsic Memory Agents
AssistantAgent can be extended to a custom class, so we can add any additional features we want to use.

In [110]:
class MemoryAgent(BaseChatAgent):

    DEFAULT_MEMORY_UPDATE_PROMPT = """Use the entire history of the groupchat (presented before this message) to 
    populate and update the current memory json with factual information. 

    *** OUTPUT SHOULD ONLY BE VALID JSON.  
    Be very careful to not include anything that renders the output not directly loadable with json.loads(). *** 
    
    For context, current memory: {memory}. 

    Newest response by yourself: {new_response}
    
    Updated memory in JSON format: """

    DEFAULT_MEMORY_REPLY_PROMPT = """Using the above instruction, group chat history, and memory json with important information from the chat history, to solve the conversation delegation agent's task. 

    Respond according to the ***last instruction from Conversation delegation agent***. 

    For context, current memory from chat based on your own outputs: {memory}. 
    
    Newest instruction: {instruction}. 
    
    Output: """

    def __init__(
        self,
        # structured_output: BaseModel,
        memory_update_prompt: Optional[str] = None,
        memory_reply_prompt: Optional[str] = None,

        *args,
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.strucutred_output = structured_output
        #self.memory_json = self.strucutred_output.model_dump(mode='json')
        self.memory_json = self.strucutred_output().model_dump()
        self.memory_update_prompt = (
            memory_update_prompt
            if memory_update_prompt != None
            else self.DEFAULT_MEMORY_UPDATE_PROMPT
        )
        self.memory_reply_prompt = (
            memory_reply_prompt
            if memory_reply_prompt != None
            else self.DEFAULT_MEMORY_REPLY_PROMPT
        )
        self.replace_reply_func(
            ConversableAgent.generate_oai_reply, MemoryAgent.generate_oai_reply
        )
        strucutred_output_config = self.llm_config.copy()
        strucutred_output_config['config_list'][0].response_format = self.strucutred_output

        self.client_memory = OpenAIWrapper(**strucutred_output_config)



    def memory_to_structured_output(self) -> BaseModel:
        from pydantic import create_model
        def dict_model(name:str,dict_def:dict):
            fields = {}
            for field_name,value in dict_def.items():
                if isinstance(value,tuple):
                    fields[field_name]=value
                elif isinstance(value,dict):
                    fields[field_name]=(dict_model(f'{name}_{field_name}',value),...)
                else:
                    raise ValueError(f"Field {field_name}:{value} has invalid syntax")
            return create_model(name,**fields)

        model = dict_model("memory",self.memory)
        return model

    @property
    def memory(self) -> dict:
        """Return the system message."""
        return self.memory_json

    @property
    def memory_prompt(self) -> str:
        """Return the system message."""
        return self.memory_prompt

    async def on_messages(self, messages, cancellation_token):
        async for message in self.on_messages_stream(messages, cancellation_token):
            if isintance(message, Response):
                return message
        raise AssertionError("The stream should have returned the final result.")

    async def on_messages_stream(self, messages, cancellation_token):
        # 1. add messages to context

        # 2. update model context with relevant memory

        # 3. generate message id for correlation between streaming chunks and final message

        # 4. run the first inference

        # 5. process the model's output

        


    def generate_oai_reply(
        self,
        messages: Optional[List[Dict]] = None,
        sender: Optional[Agent] = None,
        config: Optional[OpenAIWrapper] = None,
    ) -> Tuple[bool, Union[str, Dict, None]]:
        import pdb

        """Generate a reply using autogen.oai."""
        client = self.client if config is None else config
        if client is None:
            return False, None
        if messages is None:
            messages = self._oai_messages[sender]
            
        memory_instruction = self.memory_reply_prompt.format(
            memory=self.memory_json, instruction=messages[-1]['content']
        )
        memory_instruction = [{"content": memory_instruction, "role": "user"}]
        iostream = IOStream.get_default()

        iostream.print(f'memory response: {memory_instruction}')
        #iostream.print(f'messages: {messages}')

        extracted_response = self._generate_oai_reply_from_client(
            client,
            self._oai_system_message + messages[:-1] + memory_instruction,
            self.client_cache,
            
        )
        #iostream.print(f'extracted_response: {extracted_response}')
        instruction = [
            {
                "content": self.memory_update_prompt.format(
                    memory=self.memory_json, new_response=extracted_response
                ),
                "name": self.name,
                "role": "user",
            }
        ]
        memory_response = self._generate_oai_reply_from_client(
            self.client_memory, messages + instruction, self.client_cache
        )
        #iostream.print(f'instruction: {instruction}')

        iostream.print(colored("***** raw Memory *****", "green"), flush=True)
        iostream.print(memory_response, flush=True)

        # pattern = regex.compile(r'\{(?:[^{}]|(?R))*\}')
        # a = pattern.findall(memory_response.strip())
    
        try:
            self.memory_json = json.loads(memory_response)
        # print the message received
        except:
            iostream.print(colored("***** loaded Memory *****", "green"), flush=True)
            iostream.print("illegal format", flush=True)
        else:
            iostream.print(colored("***** loaded Memory *****", "green"), flush=True)
            iostream.print(self.memory_json, flush=True)

        return (
            (False, None) if extracted_response is None else (True, extracted_response)
        )
    async def a_generate_oai_reply(
        self,
        messages: Optional[List[Dict]] = None,
        sender: Optional[Agent] = None,
        config: Optional[Any] = None,
    ) -> Tuple[bool, Union[str, Dict, None]]:
        """Generate a reply using autogen.oai asynchronously."""
        iostream = IOStream.get_default()
        parent_context = contextvars.copy_context()

        def _generate_oai_reply(
            self, iostream: IOStream, *args: Any, **kwargs: Any
        ) -> Tuple[bool, Union[str, Dict, None]]:
            with IOStream.set_default(iostream):
                return self.generate_oai_reply(*args, **kwargs)

        memory_instruction = self.memory_reply_prompt.format(memory=self.memory_json)

        return await asyncio.get_event_loop().run_in_executor(
            None,
            lambda: parent_context.run(
                _generate_oai_reply,
                self=self,
                iostream=iostream,
                messages=messages + memory_instruction,
                sender=sender,
                config=config,
            ),
        )

NameError: name 'BaseModel' is not defined

# Customising Autogen - Group Chat